# AdaptiveTutor AI â€” GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/http-pruthvi/adaptive-tutor-ai/blob/main/training.ipynb)

**Meta PyTorch OpenEnv Hackathon Grand Finale â€” April 2026**

This notebook trains an AI tutor using GRPO (Group Relative Policy Optimization) via TRL.
The environment is running live on HuggingFace Spaces, so there's nothing to install locally.

We hit the API, collect trajectories, and use the rewards to train a small LLM to make
better tutoring decisions.

## 1. Install everything we need
torch is sometimes missing on fresh Colab runtimes, so we install it explicitly.

In [1]:
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install -q trl transformers datasets accelerate bitsandbytes
%pip install -q requests matplotlib pandas pydantic

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\pruthvi\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\torch\\_inductor\\runtime\\compile_tasks.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Connect to the live environment
Our OpenEnv server is deployed on HuggingFace Spaces. We just talk to it over HTTP.

In [2]:
import requests, json, random, time

BASE_URL = 'https://http-pruthvi-adaptive-tutor-ai.hf.space'

def env_reset():
    """Start a fresh tutoring episode."""
    r = requests.post(f'{BASE_URL}/reset', timeout=30)
    r.raise_for_status()
    data = r.json()
    return data.get('observation', data)

def env_step(action):
    """Send an action, get back observation + reward."""
    r = requests.post(f'{BASE_URL}/step', json={'action': action}, timeout=30)
    r.raise_for_status()
    data = r.json()
    obs = data.get('observation', data)
    reward = data.get('reward', obs.get('reward', 0))
    done = data.get('done', obs.get('done', False))
    info = data.get('info', obs.get('info', {}))
    return obs, reward, done, info

# quick smoke test
obs = env_reset()
print('Connected! Subject:', obs.get('current_topic', 'N/A'))
print('Student mastery:', obs.get('student_profile', {}).get('overall_mastery', 0))

Connected! Subject: math
Student mastery: 0.56


## 3. Baseline: heuristic agent
Before we train anything, let's see how a simple rule-based agent does.
This gives us a number to beat.

In [3]:
def heuristic_action(obs):
    """Simple rules: lower difficulty when struggling, raise when on a streak."""
    p = obs.get('student_profile', {})
    w = p.get('weakest_concept', '')
    cw = obs.get('consecutive_wrong', 0)
    cc = obs.get('consecutive_correct', 0)
    d = obs.get('current_difficulty', 1)
    if cw >= 2 and d > 1:
        return {'action_type': 'decrease_difficulty', 'content': '', 'target_concept': w, 'difficulty': max(1, d-1)}
    if cc >= 3:
        return {'action_type': 'increase_difficulty', 'content': '', 'target_concept': w, 'difficulty': min(5, d+1)}
    return {'action_type': 'ask_question', 'content': '', 'target_concept': w, 'difficulty': d}

def run_episode(agent_fn):
    """Run a full 20-step episode and return total reward."""
    obs = env_reset()
    total, rewards = 0.0, []
    for _ in range(20):
        action = agent_fn(obs)
        obs, reward, done, info = env_step(action)
        total += reward
        rewards.append(reward)
        if done: break
    return total, rewards

# run 5 episodes to get a stable baseline
baseline_totals = []
for i in range(5):
    total, _ = run_episode(heuristic_action)
    baseline_totals.append(total)
    print(f'Baseline episode {i+1}: {total:.1f}')

avg_baseline = sum(baseline_totals) / len(baseline_totals)
print(f'\nBaseline average: {avg_baseline:.1f}')

Baseline episode 1: 9.9
Baseline episode 2: 19.5
Baseline episode 3: 28.7
Baseline episode 4: 4.1
Baseline episode 5: 12.5

Baseline average: 14.9


## 4. Load the model
TinyLlama 1.1B â€” small enough to fit on Colab's free T4 GPU.
We'll fine-tune it to make better tutoring decisions.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map='auto'
)
print(f'Loaded {MODEL}')
print(f'{model.num_parameters()/1e6:.0f}M parameters on {model.device}')

c:\Users\pruthvi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 152.75it/s]


Loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0
1100M parameters on cuda:0


## 5. Generate training prompts
We create 60 diverse prompts by resetting the environment and stepping
to random states. This way the model sees a variety of student situations.

In [5]:
from datasets import Dataset

def build_prompt(obs):
    p = obs.get('student_profile', {})
    weak = ', '.join(p.get('weak_concepts', [])[:3]) or 'none yet'
    expert = obs.get('expert_feedback') or 'No feedback'
    return (f'You are an AI tutor deciding what to do next.\n'
            f'Subject: {obs.get("current_topic", "?")}\n'
            f'Difficulty: {obs.get("current_difficulty", 1)}/5\n'
            f'Last answer correct: {obs.get("student_correct", False)}\n'
            f'Correct streak: {obs.get("consecutive_correct", 0)}\n'
            f'Wrong streak: {obs.get("consecutive_wrong", 0)}\n'
            f'Overall mastery: {p.get("overall_mastery", 0):.0%}\n'
            f'Weak areas: {weak}\n'
            f'Expert feedback: {expert}\n\n'
            f'Respond with a JSON action: {{"action_type": "ask_question|give_hint|explain_concept|increase_difficulty|decrease_difficulty", "target_concept": "...", "difficulty": 1-5}}')

prompts = []
for i in range(60):
    obs = env_reset()
    # step a random number of times to get varied states
    for _ in range(random.randint(0, 8)):
        obs, _, done, _ = env_step(heuristic_action(obs))
        if done: break
    prompts.append({'prompt': build_prompt(obs)})
    if (i+1) % 20 == 0:
        print(f'Generated {i+1}/60 prompts')

dataset = Dataset.from_list(prompts)
print(f'\nDataset ready: {len(dataset)} prompts')

Generated 20/60 prompts
Generated 40/60 prompts
Generated 60/60 prompts

Dataset ready: 60 prompts


## 6. Define the reward function
This is where the magic happens. For each LLM completion, we parse it as
a tutoring action, run 5 steps against the live environment, and return
the cumulative reward. That becomes the GRPO training signal.

In [6]:
def parse_action(text, obs):
    """Try to pull a JSON action out of the LLM's response."""
    try:
        s, e = text.find('{'), text.rfind('}') + 1
        if s >= 0 and e > s:
            a = json.loads(text[s:e])
            return {
                'action_type': a.get('action_type', 'ask_question'),
                'content': a.get('content', ''),
                'target_concept': a.get('target_concept', ''),
                'difficulty': max(1, min(5, int(a.get('difficulty', 1))))
            }
    except:
        pass
    # if parsing fails, fall back to the heuristic
    return heuristic_action(obs)

def reward_fn(completions, **kwargs):
    """GRPO reward: run each completion against the environment."""
    rewards = []
    for comp in completions:
        text = comp if isinstance(comp, str) else comp[0].get('content', '')
        obs = env_reset()
        total = 0.0
        for _ in range(5):
            action = parse_action(text, obs)
            obs, r, done, _ = env_step(action)
            total += r
            if done: break
        rewards.append(total)
    return rewards

print('Reward function ready')

Reward function ready


## 7. Train with GRPO
Conservative settings so it actually finishes on Colab free tier.
1 epoch, batch size 2, 4 generations per prompt.

In [7]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='./grpo_output',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_completion_length=128,
    num_generations=4,
    logging_steps=5,
    save_steps=50,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    config=config,
    tokenizer=tokenizer,
    train_dataset=dataset,
    reward_funcs=reward_fn,
)

print('Starting GRPO training...')
trainer.train()
print('Done!')

RuntimeError: Failed to import trl.trainer.grpo_trainer because of the following error (look up to see its traceback):
'charmap' codec can't decode byte 0x81 in position 932: character maps to <undefined>

## 8. Evaluate the trained model
Same setup as baseline â€” 5 episodes, but now using the trained model's decisions.

In [ ]:
def trained_action(obs):
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, temperature=0.7, do_sample=True)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return parse_action(text, obs)

trained_totals = []
for i in range(5):
    total, _ = run_episode(trained_action)
    trained_totals.append(total)
    print(f'Trained episode {i+1}: {total:.1f}')

avg_trained = sum(trained_totals) / len(trained_totals)
print(f'\nTrained average: {avg_trained:.1f}')
print(f'Baseline average: {avg_baseline:.1f}')
print(f'Improvement: {avg_trained - avg_baseline:+.1f}')

## 9. Plot the results
Side-by-side comparison for the judges.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# averages
bars = ax1.bar(['Baseline\n(Heuristic)', 'Trained\n(GRPO)'],
               [avg_baseline, avg_trained],
               color=['#667eea', '#4ade80'], width=0.5, edgecolor='white', lw=1.5)
for b, v in zip(bars, [avg_baseline, avg_trained]):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}',
             ha='center', fontweight='bold', fontsize=14)
ax1.set_ylabel('Avg Episode Reward')
ax1.set_title('Before vs After Training', fontweight='bold')
ax1.set_ylim(0, max(avg_baseline, avg_trained)*1.3)
ax1.grid(axis='y', alpha=0.3)

# per-episode
ax2.plot(range(1,6), baseline_totals, 'o-', color='#667eea', label='Baseline', lw=2, ms=8)
ax2.plot(range(1,6), trained_totals, 's-', color='#4ade80', label='Trained', lw=2, ms=8)
ax2.axhline(avg_baseline, color='#667eea', ls='--', alpha=0.4)
ax2.axhline(avg_trained, color='#4ade80', ls='--', alpha=0.4)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Total Reward')
ax2.set_title('Per-Episode Comparison', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reward_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reward_comparison.png')

## 10. Push to HuggingFace Hub

In [ ]:
REPO = 'http-pruthvi/adaptive-tutor-tinyllama-grpo'

model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)
print(f'Pushed to https://huggingface.co/{REPO}')